# Framework comparison: DSPy, LangChain, Pydantic AI, OpenAI Agents SDK, and CrewAI

This notebook implements the chapter's same two-tool question-answering agent in five frameworks. Every example uses `gpt-5.6-luna` so the comparison does not change models between frameworks.

**Required environment variable:** `OPENAI_API_KEY`.

The OpenAI Agents SDK cell uses its asynchronous runner because Jupyter already owns an event loop; `Runner.run_sync` remains appropriate in a normal synchronous Python script.


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))


## Shared tools


In [ ]:
def search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy is a framework for programming language models. Query: {query}"


def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


## 1. DSPy


In [ ]:
dspy_agent = dspy.ReAct(
    "question -> answer",
    tools=[search_knowledge_base, calculate],
    max_iters=5,
)
dspy_result = dspy_agent(question="What is DSPy and what is 15 * 23?")
print(dspy_result.answer)


## 2. LangChain agent runtime (built on LangGraph)


In [ ]:
%pip install 'langchain>=1' langchain-openai -q


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool


@tool
def lc_search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy is a framework for programming language models. Query: {query}"


@tool
def lc_calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


lc_model = ChatOpenAI(model="gpt-5.6-luna")
lc_agent = create_agent(lc_model, [lc_search_knowledge_base, lc_calculate])
lc_result = lc_agent.invoke(
    {"messages": [{"role": "user", "content": "What is DSPy and what is 15 * 23?"}]}
)
print(lc_result["messages"][-1].content)


## 3. Pydantic AI


In [ ]:
%pip install pydantic-ai -q


In [ ]:
from pydantic_ai import Agent as PydanticAgent


pydantic_agent = PydanticAgent("openai:gpt-5.6-luna")


@pydantic_agent.tool_plain
def pydantic_search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy is a framework for programming language models. Query: {query}"


@pydantic_agent.tool_plain
def pydantic_calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


pydantic_result = pydantic_agent.run_sync(
    "What is DSPy and what is 15 * 23?"
)
print(pydantic_result.output)


## 4. OpenAI Agents SDK


In [ ]:
%pip install openai-agents -q


In [ ]:
from agents import Agent as OpenAIAgent, Runner, function_tool


@function_tool
def openai_search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy is a framework for programming language models. Query: {query}"


@function_tool
def openai_calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


openai_agent = OpenAIAgent(
    name="QA Agent",
    instructions="Answer questions using available tools.",
    model="gpt-5.6-luna",
    tools=[openai_search_knowledge_base, openai_calculate],
)
openai_result = await Runner.run(
    openai_agent,
    "What is DSPy and what is 15 * 23?",
)
print(openai_result.final_output)


## 5. CrewAI


In [ ]:
%pip install crewai -q


In [ ]:
from crewai import Agent as CrewAgent, Task, Crew
from crewai.tools import tool as crew_tool


@crew_tool("Search Knowledge Base")
def crew_search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy is a framework for programming language models. Query: {query}"


@crew_tool("Calculator")
def crew_calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


crew_agent = CrewAgent(
    role="Research Assistant",
    goal="Answer questions accurately using available tools",
    backstory="You are a helpful assistant.",
    llm="gpt-5.6-luna",
    tools=[crew_search_knowledge_base, crew_calculate],
)
crew_task = Task(
    description="What is DSPy and what is 15 * 23?",
    expected_output="A clear answer",
    agent=crew_agent,
)
crew = Crew(agents=[crew_agent], tasks=[crew_task])
crew_result = crew.kickoff()
print(crew_result)
